# 03 — Train & Compare MLP (GPU)

**Mục tiêu:** train nhiều cấu hình MLP độc lập từ cùng processed dataset Pose13 V2; chọn bằng validation, chỉ đánh giá test khi chạy Final. Toàn bộ features nhỏ được đọc **một lần** và giữ trên GPU.  
**Đầu ra:** mỗi lần chạy có thư mục `_v001`, `_v002`, ... không ghi đè; mỗi candidate lưu checkpoint, metrics và history.  
**Quan trọng:** `test_as_val` dùng test để chọn model nên **không còn test độc lập**.

## Cell 1 — Cấu hình & môi trường GPU
Chọn input ZIP, tên thư mục output, danh sách validation và các candidates. Candidate chỉ ghi biến muốn thay đổi; tất cả còn lại dùng `DEFAULT`. Mặc định `RUN_FINAL=True`, `FINAL_CANDIDATE="auto"`: chạy hết candidates → chọn theo validation → tự retrain Final → đánh giá test. Không phải sửa Cell 9.

In [ ]:
import os
# Đặt trước import torch để hỗ trợ deterministic CUDA matmul.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import sys, io, json, zipfile, shutil, hashlib, random, re, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
import torch
from torch import nn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

COLAB_ROOT = Path("/content/drive/MyDrive/KI_8/IMP302/LAB_2/truongedu")
ROOT = Path(os.environ["POSEFORGE_ROOT"]).expanduser() if os.environ.get("POSEFORGE_ROOT") else (
    COLAB_ROOT if COLAB_ROOT.is_dir() else next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / "notebooks").is_dir() and (p / "shared").is_dir()), Path.cwd()
    )
)
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from shared.mlp_model import PostureMLP, MODEL_REGISTRY

PROCESSED_ZIP = Path(os.environ.get(
    "POSEFORGE_PROCESSED_ZIP",
    str(ROOT / "outputs/poseforge_pose13_v2/poseforge_pose13_v2_test_TRUONG.zip")
))
OUTPUT_NAME = "mlp_experiment"
TEST_PEOPLE = ["TRUONG"]   # Phải đúng với processed ZIP (không đổi split ở đây)
VAL_PEOPLE = ["CHANH", "KIET"]
VAL_MODE = "separate"       # separate | test_as_val | none
RUN_FINAL = True            # Run all: train candidates → auto Final → test
FINAL_CANDIDATE = "auto"   # auto | tên candidate muốn chỉ định

DEFAULT = {
    "model": "mlp_base", "seed": 302,
    "lr": 1e-3, "weight_decay": 1e-3, "batch_size": 256,
    "epochs": 80, "patience": 12, "dropout": 0.15,
    "max_frames_per_clip": 48,
    "flip_prob": 0.0, "jitter_prob": 0.0, "jitter_sigma": 0.015,
    "keypoint_dropout_prob": 0.0,
}
CANDIDATES = [
    {"name": "baseline"},
    {"name": "flip_jitter", "flip_prob": 0.5, "jitter_prob": 0.5},
    {"name": "mlp_wide", "model": "mlp_wide", "dropout": 0.20},
]

DEVICE_NAME = os.environ.get("POSEFORGE_DEVICE", "cuda")  # GPU mặc định; cpu chỉ để test
assert DEVICE_NAME == "cpu" or torch.cuda.is_available(), (
    "CUDA không khả dụng. Colab: Runtime → Change runtime type → GPU."
)
DEVICE = torch.device(DEVICE_NAME)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

names = [c["name"] for c in CANDIDATES]
assert len(names) == len(set(names)) and all(re.fullmatch(r"[a-zA-Z0-9_-]+", n) for n in names)
assert re.fullmatch(r"[a-zA-Z0-9_-]+", OUTPUT_NAME)
assert VAL_MODE in {"separate", "test_as_val", "none"}
assert len(TEST_PEOPLE) == len(set(TEST_PEOPLE)) and TEST_PEOPLE
for c in CANDIDATES:
    cfg = {**DEFAULT, **c}
    assert cfg["model"] in MODEL_REGISTRY and 0 <= cfg["dropout"] < 1
    assert cfg["epochs"] > 0 and cfg["batch_size"] > 0 and cfg["patience"] > 0
    for key in ("flip_prob", "jitter_prob", "keypoint_dropout_prob"):
        assert 0 <= cfg[key] <= 1, f"Invalid {key} in {c['name']}"
assert PROCESSED_ZIP.is_file(), f"Missing processed ZIP: {PROCESSED_ZIP}"
print("Root:", ROOT, " | Dataset:", PROCESSED_ZIP)
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU smoke-test")
print("Candidates:", names, "| VAL_MODE:", VAL_MODE, "| RUN_FINAL:", RUN_FINAL)

## Cell 2 — Đọc processed ZIP một lần
Kiểm tra `feature_contract.json`, `index.csv` và từng NPZ `[T,13,4]`; chỉ nạp `x` (không nạp MP4/raw17). Copy ZIP sang runtime Colab để tránh đọc Drive nhiều lần.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

SOURCE_SHA = sha256_file(PROCESSED_ZIP)
if Path("/content").exists() and PROCESSED_ZIP.resolve().as_posix().startswith("/content/drive/"):
    local_zip = Path("/content") / f"poseforge_processed_{SOURCE_SHA[:12]}.zip"
    if not local_zip.exists() or sha256_file(local_zip) != SOURCE_SHA:
        with PROCESSED_ZIP.open("rb") as src, local_zip.open("wb") as dst, tqdm(
            total=PROCESSED_ZIP.stat().st_size, unit="B", unit_scale=True, desc="Copy ZIP to runtime"
        ) as bar:
            while (chunk := src.read(4 * 1024 * 1024)):
                dst.write(chunk); bar.update(len(chunk))
    assert sha256_file(local_zip) == SOURCE_SHA
else:
    local_zip = PROCESSED_ZIP

with zipfile.ZipFile(local_zip) as z:
    prefix = next((n[:-len("index.csv")] for n in z.namelist() if n.endswith("/index.csv")), None)
    assert prefix is not None, "index.csv not found"
    index = pd.read_csv(io.BytesIO(z.read(prefix + "index.csv")))
    feature_contract = json.loads(z.read(prefix + "feature_contract.json"))
    split_manifest = json.loads(z.read(prefix + "split_manifest.json"))
    assert feature_contract["shape_per_frame"] == [13, 4]
    assert feature_contract["flatten_dim"] == 52
    assert feature_contract["extractor"] == "yolo11n-pose.pt"
    assert index.clip_id.is_unique and len(index) > 0
    x_parts, y_parts, clip_info = [], [], []
    offset = 0
    for row in tqdm(index.itertuples(index=False), total=len(index), desc="Load 52-feature clips"):
        with np.load(io.BytesIO(z.read(prefix + row.feature_path)), allow_pickle=False) as npz:
            x = npz["x"].astype("float32", copy=False)
        assert x.ndim == 3 and x.shape[1:] == (13, 4) and len(x) == int(row.frames)
        assert np.isfinite(x).all() and np.isin(x[:, :, 3], (0, 1)).all(), row.clip_id
        x_parts.append(x.reshape(len(x), 52))
        y_parts.append(np.full(len(x), int(row.label), dtype="int64"))
        clip_info.append({"clip_id": row.clip_id, "person": row.person_id,
                          "camera": row.camera_id, "label": int(row.label), "split": row.split,
                          "start": offset, "end": offset + len(x)})
        offset += len(x)

X = np.concatenate(x_parts).astype("float32", copy=False)
Y = np.concatenate(y_parts)
assert len(X) == len(Y) == offset and set(np.unique(Y)) == set(range(6))
X_GPU = torch.from_numpy(X).to(DEVICE)
Y_GPU = torch.from_numpy(Y).to(DEVICE)
LABEL_NAMES = ["Upright", "Hunched", "Head on desk", "Leaning back", "Side lean/turn", "Hand raised"]
print(f"{len(clip_info)} clips | {len(X):,} frames | X {X.shape} | dataset SHA256 {SOURCE_SHA[:16]}...")
print("Feature contract:", feature_contract["version"], "| extractor:", feature_contract["extractor"])

## Cell 3 — Khóa train/validation/test theo người
Từ `train_pool` của Notebook 02: chọn `VAL_PEOPLE` hoặc `test_as_val`. **Không trộn frame của cùng người sang train và validation/test.** Test chỉ dùng ở Cell 9 khi `RUN_FINAL=True` (trừ chế độ `test_as_val`).

In [ ]:
test_people_in_zip = sorted({c["person"] for c in clip_info if c["split"] == "test"})
train_pool_people = sorted({c["person"] for c in clip_info if c["split"] == "train_pool"})
assert sorted(TEST_PEOPLE) == test_people_in_zip, (
    f"TEST_PEOPLE mismatch: config={TEST_PEOPLE}, ZIP={test_people_in_zip}"
)
assert not (set(test_people_in_zip) & set(train_pool_people))
if VAL_MODE == "separate":
    assert VAL_PEOPLE and len(set(VAL_PEOPLE)) == len(VAL_PEOPLE)
    assert set(VAL_PEOPLE) <= set(train_pool_people), "Validation must be inside train_pool"
    train_people = set(train_pool_people) - set(VAL_PEOPLE)
    val_people = set(VAL_PEOPLE)
elif VAL_MODE == "test_as_val":
    train_people, val_people = set(train_pool_people), set(test_people_in_zip)
    print("[WARNING] Test is used for model selection: no independent test score.")
else:
    train_people, val_people = set(train_pool_people), set()
assert train_people and not train_people & val_people

train_clips = [c for c in clip_info if c["person"] in train_people]
val_clips = [c for c in clip_info if c["person"] in val_people]
test_clips = [c for c in clip_info if c["split"] == "test"]
train_pool_clips = [c for c in clip_info if c["split"] == "train_pool"]

def frame_indices(clips):
    return np.concatenate([np.arange(c["start"], c["end"], dtype="int64") for c in clips]) if clips else np.empty(0, dtype="int64")

def count_by_class(clips):
    return np.bincount(Y[frame_indices(clips)], minlength=6)

split_table = pd.DataFrame([
    {"split": label, "people": len(set(c["person"] for c in clips)),
     "clips": len(clips), "frames": sum(c["end"]-c["start"] for c in clips),
     **{f"L{i}": int(v) for i, v in enumerate(count_by_class(clips))}}
    for label, clips in (("train", train_clips), ("val", val_clips), ("test", test_clips))
])
display(split_table)
assert np.all(count_by_class(train_clips) > 0), "Training missing a class"
if val_clips: assert np.all(count_by_class(val_clips) > 0), "Validation missing a class"
assert np.all(count_by_class(test_clips) > 0), "Test missing a class"
assert not set(c["clip_id"] for c in train_clips) & set(c["clip_id"] for c in test_clips)
print("Train:", sorted(train_people), " | Validation:", sorted(val_people), " | Frozen test:", test_people_in_zip)

## Cell 4 — Sampling và augmentation trên GPU
Mỗi epoch chọn tối đa 48 frame/clip-camera (lấy lại mẫu mỗi epoch). Flip hoán đổi trái/phải; jitter chỉ tác động tọa độ còn `mask=1`. Random riêng cho từng epoch/candidate; validation/test không augment.

In [ ]:
# Thứ tự 13 joints COCO: nose, eyes L/R, ears L/R, shoulders L/R, elbows L/R, wrists L/R, hips L/R.
FLIP_INDEX = torch.tensor([0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11], device=DEVICE)

def sample_epoch(clips, max_frames, seed, epoch):
    sample_rng = np.random.default_rng(np.random.SeedSequence([int(seed), epoch, 101]))
    items = []
    for clip in clips:
        ids = np.arange(clip["start"], clip["end"], dtype=np.int64)
        count = min(len(ids), int(max_frames))
        items.append(sample_rng.choice(ids, size=count, replace=False) if len(ids) > count else ids)
    result = np.concatenate(items)
    np.random.default_rng(np.random.SeedSequence([int(seed), epoch, 202])).shuffle(result)
    return result

def augment_batch(x, cfg, rng):
    if not any(cfg[k] > 0 for k in ("flip_prob", "jitter_prob", "keypoint_dropout_prob")):
        return x
    a = x.reshape(-1, 13, 4).clone()
    n = len(a)
    flip = torch.rand(n, device=DEVICE, generator=rng) < cfg["flip_prob"]
    if flip.any():
        a[flip] = a[flip][:, FLIP_INDEX]
        a[flip, :, 0] *= -1
    jitter = (torch.rand(n, 1, 1, device=DEVICE, generator=rng) < cfg["jitter_prob"])
    if cfg["jitter_prob"] > 0:
        noise = torch.randn((n, 13, 2), device=DEVICE, generator=rng) * cfg["jitter_sigma"]
        a[:, :, :2] += noise * jitter * a[:, :, 3:4]
    if cfg["keypoint_dropout_prob"] > 0:
        dropped = (torch.rand((n, 13), device=DEVICE, generator=rng) < cfg["keypoint_dropout_prob"])
        a[dropped] = 0
    return a.reshape(-1, 52)

# Contract: missing joints stay exactly zero; mirrored x changes sign, left/right indices swap.
example = X_GPU[:min(8, len(X_GPU))]
assert torch.equal(augment_batch(example, {**DEFAULT, "flip_prob":0, "jitter_prob":0},
                                 torch.Generator(device=DEVICE).manual_seed(1)), example)
print("Sampling: dynamic / epoch; augmentation: GPU, train only; no separate augmented dataset.")

## Cell 5 — Đánh giá & checkpoint
Frame-level Accuracy, Macro-F1, Loss; Clip-level Macro-F1 bằng trung bình xác suất trong từng clip. Ghi đúng model state, feature contract và config để Phase 3 nạp lại.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def predict_clips(model, clips, batch_size=4096):
    model.eval()
    ids = frame_indices(clips)
    if not len(ids): return None
    probs = []
    for block in np.array_split(ids, max(1, int(np.ceil(len(ids) / batch_size)))):
        probs.append(model(X_GPU[torch.as_tensor(block, device=DEVICE)]).softmax(dim=1).cpu().numpy())
    p = np.concatenate(probs)
    y = Y[ids]
    pred = p.argmax(axis=1)
    clip_true, clip_pred, camera_rows, person_rows = [], [], [], []
    cursor = 0
    for c in clips:
        n = c["end"] - c["start"]
        cp = p[cursor:cursor+n].mean(axis=0).argmax()
        clip_true.append(c["label"]); clip_pred.append(int(cp))
        camera_rows.extend([c["camera"]] * n)
        person_rows.extend([c["person"]] * n)
        cursor += n
    return {"y": y, "pred": pred, "probs": p, "clip_y": np.array(clip_true),
            "clip_pred": np.array(clip_pred), "camera": np.array(camera_rows),
            "person": np.array(person_rows)}

def score_prediction(result):
    if result is None: return {}
    return {"accuracy": float(accuracy_score(result["y"], result["pred"])),
            "macro_f1": float(f1_score(result["y"], result["pred"], labels=list(range(6)), average="macro", zero_division=0)),
            "clip_accuracy": float(accuracy_score(result["clip_y"], result["clip_pred"])),
            "clip_macro_f1": float(f1_score(result["clip_y"], result["clip_pred"], labels=list(range(6)), average="macro", zero_division=0))}

def detailed_report(result, dest, title):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    metrics = score_prediction(result)
    report = classification_report(result["y"], result["pred"], labels=list(range(6)),
                                   target_names=LABEL_NAMES, output_dict=True, zero_division=0)
    (dest / "metrics.json").write_text(json.dumps(metrics, indent=2))
    pd.DataFrame(report).transpose().to_csv(dest / "per_class_metrics.csv")
    cm = confusion_matrix(result["y"], result["pred"], labels=list(range(6)))
    fig, ax = plt.subplots(figsize=(7.5, 6))
    im = ax.imshow(cm, cmap="Blues")
    for (i,j),v in np.ndenumerate(cm): ax.text(j, i, str(v), ha="center", va="center", fontsize=8)
    ax.set_xticks(range(6), [f"L{i}" for i in range(6)])
    ax.set_yticks(range(6), [f"L{i}" for i in range(6)])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(dest / "confusion_matrix.png", dpi=150); plt.close(fig)
    groups = []
    for field in ("camera", "person"):
        for g in sorted(set(result[field])):
            keep = result[field] == g
            groups.append({"group_type":field, "group":g, "frames": int(keep.sum()),
                "accuracy":float(accuracy_score(result["y"][keep], result["pred"][keep])),
                "macro_f1":float(f1_score(result["y"][keep], result["pred"][keep],labels=list(range(6)),average="macro",zero_division=0))})
    pd.DataFrame(groups).to_csv(dest / "metrics_by_camera_person.csv", index=False)
    return metrics

def checkpoint(model, cfg, epoch, people, stage, metrics=None):
    return {"model_state_dict": {k: v.detach().cpu().clone() for k,v in model.state_dict().items()},
            "model_name": cfg["model"], "model_config": cfg,
            "feature_contract": feature_contract, "label_names": LABEL_NAMES,
            "source_processed_zip_sha256": SOURCE_SHA,
            "train_people": sorted(people), "epoch": int(epoch), "stage": stage,
            "metrics": metrics or {}}

print("Models:", MODEL_REGISTRY, "| Features:", 52, "| Classes:", 6)

## Cell 6 — Hàm train một candidate
Khởi tạo lại model, optimizer, seed và random generator. Early stopping theo **Validation Macro-F1** (tie-break Validation Loss), không truy cập test khi `separate`. Train metrics đo trên dữ liệu gốc không augmentation.

In [ ]:
def eval_loss(result):
    p = np.clip(result["probs"][np.arange(len(result["y"])), result["y"]], 1e-9, 1)
    return float(-np.log(p).mean())

def run_candidate(cfg, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=False)
    (dest / "config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    seed_everything(int(cfg["seed"]))  # độc lập với mọi candidate chạy trước
    model = PostureMLP(cfg["model"], dropout=cfg["dropout"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()
    best_f1, best_loss, best_epoch, stale = -1., float("inf"), 0, 0
    history = []
    progress = tqdm(range(1, int(cfg["epochs"])+1), desc=cfg["name"], leave=True)
    for epoch in progress:
        model.train()
        ids = sample_epoch(train_clips, cfg["max_frames_per_clip"], cfg["seed"], epoch)
        aug_rng = torch.Generator(device=DEVICE)
        aug_rng.manual_seed(int(cfg["seed"])*1_000_003 + epoch*1_001 + 17)
        batch_losses = []
        for chunk in np.array_split(ids, max(1, int(np.ceil(len(ids) / cfg["batch_size"])))):
            idx = torch.as_tensor(chunk, device=DEVICE)
            xb = augment_batch(X_GPU[idx], cfg, aug_rng)
            logits = model(xb)
            loss = loss_fn(logits, Y_GPU[idx])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.detach()))
        train_result = predict_clips(model, train_clips)
        train_score = score_prediction(train_result)
        val_result = predict_clips(model, val_clips) if val_clips else None
        val_score = score_prediction(val_result) if val_result else {}
        v_loss = eval_loss(val_result) if val_result else None
        record = {"epoch": epoch, "batch_loss": float(np.mean(batch_losses)),
                  "train_loss": eval_loss(train_result), "train_f1": train_score["macro_f1"],
                  "train_accuracy": train_score["accuracy"], "val_loss": v_loss,
                  "val_f1": val_score.get("macro_f1"), "val_accuracy": val_score.get("accuracy"),
                  "val_clip_f1": val_score.get("clip_macro_f1")}
        history.append(record)
        improved = (val_result is None or val_score["macro_f1"] > best_f1 + 1e-6
                    or (abs(val_score["macro_f1"] - best_f1) <= 1e-6 and v_loss < best_loss-1e-6))
        if improved:
            best_f1 = val_score.get("macro_f1", train_score["macro_f1"])
            best_loss = v_loss if v_loss is not None else record["train_loss"]
            best_epoch, stale = epoch, 0
            torch.save(checkpoint(model, cfg, epoch, train_people, "selection", val_score), dest / "best.pt")
        else:
            stale += 1
        progress.set_postfix(train_f1=f"{train_score['macro_f1']:.3f}",
                             val_f1=f"{val_score['macro_f1']:.3f}" if val_score else "—", patience=stale)
        if val_clips and stale >= int(cfg["patience"]): break
    pd.DataFrame(history).to_csv(dest / "history.csv", index=False)
    saved = torch.load(dest / "best.pt", map_location=DEVICE, weights_only=True)
    model.load_state_dict(saved["model_state_dict"])
    model.eval()
    best_val = predict_clips(model, val_clips) if val_clips else None
    metrics = detailed_report(best_val, dest / "validation", cfg["name"] + " validation") if best_val else {}
    summary = {"candidate": cfg["name"], "model": cfg["model"], "best_epoch": best_epoch,
               "trained_epochs": len(history), "val_mode": VAL_MODE,
               "test_independent": VAL_MODE != "test_as_val", **{("val_" + k):v for k,v in metrics.items()}}
    (dest / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

print("Train function ready | model + optimizer + RNG reset each candidate.")

## Cell 7 — Train tuần tự các candidates (không ghi đè output)
Mỗi lần chạy tự tăng `_v001`, `_v002`...; candidates có folder riêng. Có thanh tiến độ từng epoch. `RUN_FINAL=True` (mặc định) sẽ chạy tiếp Final sau khi so sánh validation; đặt False nếu chỉ muốn train candidates.

In [ ]:
RUN_BASE = ROOT / "outputs/mlp_training"
RUN_BASE.mkdir(parents=True, exist_ok=True)
for version in range(1, 10_000):
    RUN_DIR = RUN_BASE / f"{OUTPUT_NAME}_v{version:03d}"
    try:
        RUN_DIR.mkdir(exist_ok=False)
        break
    except FileExistsError:
        pass
else:
    raise RuntimeError("Too many experiment versions")

(RUN_DIR / "feature_contract.json").write_text(json.dumps(feature_contract, indent=2))
(RUN_DIR / "experiment_config.json").write_text(json.dumps({
    "output_name": OUTPUT_NAME, "dataset_sha256": SOURCE_SHA,
    "test_people": test_people_in_zip, "val_people": sorted(val_people), "val_mode": VAL_MODE,
    "run_final": RUN_FINAL, "final_candidate": FINAL_CANDIDATE,
    "default": DEFAULT, "candidates": CANDIDATES
}, indent=2))
(RUN_DIR / "environment.json").write_text(json.dumps({
    "python": platform.python_version(), "torch": torch.__version__,
    "numpy": np.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU",
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG")
}, indent=2))
print("New run folder:", RUN_DIR)
results = []
for candidate in CANDIDATES:
    cfg = {**DEFAULT, **candidate}
    print(f"=== {cfg['name']} | {cfg['model']} | seed {cfg['seed']} ===")
    results.append(run_candidate(cfg, RUN_DIR / "candidates" / cfg["name"]))
comparison = pd.DataFrame(results)
if val_clips:
    comparison = comparison.sort_values(
        ["val_macro_f1", "val_clip_macro_f1", "candidate"],
        ascending=[False, False, True], kind="mergesort"
    )
comparison.to_csv(RUN_DIR / "comparison.csv", index=False)
display(comparison)
print("Candidates complete; test still untouched" if VAL_MODE != "test_as_val" else "Test used as validation: exploratory scores only")

## Cell 8 — So sánh candidates
Vẽ train/validation Loss, Macro-F1 và ranking từ cùng split; lưu PNG. Nếu `FINAL_CANDIDATE="auto"`, Cell 9 tự chọn theo Validation Macro-F1 (hòa thì dùng clip Macro-F1, sau cùng tên candidate), không dùng test.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for candidate in CANDIDATES:
    name = candidate["name"]
    hist = pd.read_csv(RUN_DIR / "candidates" / name / "history.csv")
    axes[0].plot(hist.epoch, hist.train_loss, label=name + " train")
    if val_clips: axes[0].plot(hist.epoch, hist.val_loss, linestyle="--", label=name + " val")
    axes[1].plot(hist.epoch, hist.train_f1, label=name + " train")
    if val_clips: axes[1].plot(hist.epoch, hist.val_f1, linestyle="--", label=name + " val")
axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Macro-F1", xlabel="Epoch", ylabel="F1", ylim=(0,1))
for ax in axes[:2]: ax.grid(alpha=.2); ax.legend(fontsize=7)
if val_clips:
    cmp = comparison.sort_values("val_macro_f1", ascending=False)
    axes[2].bar(cmp.candidate, cmp.val_macro_f1)
    axes[2].set(title="Validation Macro-F1", ylabel="F1", ylim=(0,1))
else:
    axes[2].text(.5,.5,"No validation: no candidate ranking", ha="center", va="center")
    axes[2].set_axis_off()
axes[2].tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(RUN_DIR / "candidate_comparison.png", dpi=150)
plt.show(); plt.close(fig)
print("Saved:", RUN_DIR / "candidate_comparison.png")

## Cell 9 — Tự chọn best candidate → Final → Test
Mặc định chạy **tự động** khi Run all: chọn candidate theo **Validation Macro-F1**; hòa thì dùng Clip Macro-F1, rồi tên. Retrain từ đầu trên đủ `train_pool` với số epoch đã chọn qua validation, sau đó đánh giá test. `FINAL_CANDIDATE="tên"` cho phép chọn thủ công nếu cần. Với `VAL_MODE="none"`, không có ranking đáng tin cậy nên mặc định dùng candidate **đầu tiên** và ghi rõ trong báo cáo. Với `test_as_val`, test đã tham gia chọn model nên kết quả **không độc lập**.

In [ ]:
final_summary = None
if not RUN_FINAL:
    print("SKIPPED final/test — set RUN_FINAL=True in Cell 1 to run automatically.")
else:
    if FINAL_CANDIDATE == "auto":
        if val_clips:
            chosen_name = str(comparison.iloc[0]["candidate"])  # đã sort theo validation
            selection_basis = "val_macro_f1 > val_clip_macro_f1 > name"
        else:
            chosen_name = CANDIDATES[0]["name"]
            selection_basis = "first_candidate (no validation)"
            print("[WARNING] No validation: choose first candidate, not best by generalization.")
    else:
        assert FINAL_CANDIDATE in names, f"Choose from {names} or 'auto'"
        chosen_name = FINAL_CANDIDATE
        selection_basis = "manual_override"
    chosen_cfg = {**DEFAULT, **next(c for c in CANDIDATES if c["name"] == chosen_name)}
    selected_summary = next(r for r in results if r["candidate"] == chosen_name)
    print(f"[FINAL] Selected {chosen_name} | {selection_basis} | epoch={selected_summary['best_epoch']}")
    config_file = RUN_DIR / "experiment_config.json"
    updated_config = json.loads(config_file.read_text())
    updated_config.update({"run_final": True, "final_candidate": chosen_name,
                           "final_selection_basis": selection_basis})
    config_file.write_text(json.dumps(updated_config, indent=2))
    final_dir = RUN_DIR / "final"
    final_dir.mkdir(exist_ok=False)
    final_epoch_count = int(selected_summary["best_epoch"])
    seed_everything(int(chosen_cfg["seed"]) + 10_000)
    model = PostureMLP(chosen_cfg["model"], chosen_cfg["dropout"]).to(DEVICE)
    if VAL_MODE == "none":
        ckpt = torch.load(RUN_DIR / "candidates" / chosen_name / "best.pt",
                          map_location=DEVICE, weights_only=True)
        model.load_state_dict(ckpt["model_state_dict"])
        print("No validation: reuse checkpoint trained on full train pool")
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=chosen_cfg["lr"],
                                weight_decay=chosen_cfg["weight_decay"])
        loss_fn = nn.CrossEntropyLoss()
        full_history = []
        for epoch in tqdm(range(1, final_epoch_count + 1), desc="Final retrain 12-person pool"):
            model.train()
            indices = sample_epoch(train_pool_clips, chosen_cfg["max_frames_per_clip"],
                                   chosen_cfg["seed"] + 10_000, epoch)
            aug_rng = torch.Generator(device=DEVICE).manual_seed(
                (int(chosen_cfg["seed"])+10_000)*1_000_003 + epoch*1_001 + 17)
            losses = []
            for chunk in np.array_split(indices, max(1,int(np.ceil(len(indices)/chosen_cfg["batch_size"])))):
                idx = torch.as_tensor(chunk, device=DEVICE)
                xb = augment_batch(X_GPU[idx], chosen_cfg, aug_rng)
                loss = loss_fn(model(xb), Y_GPU[idx])
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                losses.append(float(loss.detach()))
            full_history.append({"epoch":epoch, "train_batch_loss":float(np.mean(losses))})
        pd.DataFrame(full_history).to_csv(final_dir / "final_history.csv",index=False)
    model.eval()
    # Test is only evaluated here. In test_as_val mode it has already been used for tuning.
    test_pred = predict_clips(model, test_clips)
    test_metrics = detailed_report(test_pred, final_dir / "test", "Final test")
    independent = VAL_MODE != "test_as_val"
    final_summary = {"candidate":chosen_name, "selection_basis":selection_basis,
                     "selected_validation_macro_f1":selected_summary.get("val_macro_f1"),
                     "train_people":train_pool_people,
                     "test_people":test_people_in_zip, "epochs":final_epoch_count,
                     "test_independent":independent, "test_metrics":test_metrics,
                     "evaluation_note":("Independent held-out test" if independent
                                        else "TEST USED AS VALIDATION: exploratory only")}
    torch.save(checkpoint(model, chosen_cfg, final_epoch_count, train_pool_people,
                          "final", test_metrics), final_dir / "model_final.pt")
    (final_dir / "final_summary.json").write_text(json.dumps(final_summary, indent=2))
    print("Final:", final_summary["evaluation_note"])
    display(pd.DataFrame([test_metrics]))
    print("Model:", final_dir / "model_final.pt")

## Cell 10 — Export và kiểm tra output
Lưu bảng kết quả, config, metrics, biểu đồ, checkpoint. `review.zip` nhỏ **không chứa models/raw data**; nếu chạy Final, tạo thêm `model_bundle.zip` để deploy với feature contract và định nghĩa model.

In [ ]:
review_path = RUN_DIR / "experiment_review.zip"
with zipfile.ZipFile(review_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN_DIR.rglob("*")):
        if f.is_file() and f != review_path and f.suffix in {".json", ".csv", ".png"}:
            z.write(f, f.relative_to(RUN_DIR))

model_bundle = None
if RUN_FINAL:
    model_bundle = RUN_DIR / "final_model_bundle.zip"
    with zipfile.ZipFile(model_bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(RUN_DIR / "final/model_final.pt", "model_final.pt")
        z.write(RUN_DIR / "final/final_summary.json", "final_summary.json")
        z.write(RUN_DIR / "feature_contract.json", "feature_contract.json")
        z.write(RUN_DIR / "experiment_config.json", "experiment_config.json")
        z.write(ROOT / "shared/mlp_model.py", "shared/mlp_model.py")

print("=== OUTPUT SUMMARY ===")
print("Run:", RUN_DIR)
print("Review ZIP:", review_path, f"({review_path.stat().st_size/1024:.1f} KiB)")
if model_bundle: print("Deploy ZIP:", model_bundle)
print("Validation:", sorted(val_people), "| Frozen test:", test_people_in_zip)
print("Test independently held out:", VAL_MODE != "test_as_val")
print("Best candidate by validation:", comparison.iloc[0]["candidate"] if val_clips else "N/A (no validation)")
with zipfile.ZipFile(review_path) as z:
    assert z.testzip() is None
if model_bundle:
    with zipfile.ZipFile(model_bundle) as z:
        assert z.testzip() is None
print("[PASS] ZIP integrity; no model/data copied to review archive")